# 03 — Statistical photometric features

Construct one row per object: twelve flux statistics in each of six passbands,
fifteen cross-band and temporal summaries, and four photometric metadata fields.
The resulting block contains 91 model inputs. Preserve object order and identifiers.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))  

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src.features.statistics import build_feature_table, FEATURE_NAMES, NEW_FEATURES, COLS_META
from src.features.cache import write_cache, load_cache


## 1. Load the cleaned light curves


In [ ]:
df = pd.read_csv(config.TRAIN_CLEAN_CSV)
metadata_table = pd.read_csv(config.METADATA_CLEAN_CSV)
print(f'{df.object_id.nunique():,} objects')


## 2. Build the feature table


In [ ]:
RECOMPUTE_FEATURES = False
if RECOMPUTE_FEATURES:
    result_table = build_feature_table(df, metadata_table)
else:
    result_table = load_cache(config.FEATURES_CSV, 'estadisticas')
    print('Validated existing feature cache; extraction was not repeated.')
n_feat = result_table.shape[1] - 2  
print(f'{result_table.shape[0]:,} objects · {n_feat} features '
      f'({len(FEATURE_NAMES)*6} stats + {len(NEW_FEATURES)} new + {len(COLS_META)} metadata)')
assert result_table.isnull().sum().sum() == 0, 'The final table contains NaN values'
result_table.head()


## 3. Save the feature cache


In [ ]:
if RECOMPUTE_FEATURES:
    write_cache(result_table, config.FEATURES_CSV, 'estadisticas', producer_name='notebooks/03_statistical_features.ipynb')
    print(f'Saved: {config.FEATURES_CSV.name}')
else:
    print('Retained existing feature CSV and manifest unchanged.')


## 4. Descriptive feature separation

Compare effect sizes between selected classes. These diagnostics describe
separation in individual variables; they are not a substitute for held-out
evaluation or a causal attribution of predictive improvement.


In [ ]:
from src.features.discrimination import between_class_effect_sizes, one_vs_rest_effect_sizes, block_summary


pares_dificiles = [(52, 90), (67, 90), (62, 90), (42, 90)]


In [ ]:
for a, b in pares_dificiles:
    tab = between_class_effect_sizes(result_table, a, b)
    print(f"\n=== Class {a} vs {b} — top 10 features by |d| ===")
    print(tab.head(10).to_string(index=False))


In [ ]:
summary_table = block_summary(result_table, pares_dificiles)
print('Mean |d| by feature block:')
print(summary_table)


In [ ]:

fig, ax = plt.subplots(figsize=(7, 3.6))
data = summary_table.values.astype(float)
NAMES = {52:'SNIax', 67:'91bg', 62:'SNIbc', 42:'SNII', 90:'SNIa'}
cols_lbl = [f"{NAMES[a]} vs {NAMES[b]}" for (a, b) in pares_dificiles]
im = ax.imshow(data, cmap='YlOrRd', aspect='auto', vmin=0)
ax.set_xticks(range(len(pares_dificiles))); ax.set_xticklabels(cols_lbl, fontsize=9)
ax.set_yticks(range(len(summary_table.index))); ax.set_yticklabels(summary_table.index, fontsize=9)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f"{data[i,j]:.2f}", ha='center', va='center', fontsize=9)
ax.set_title('Mean Cohen |d| by feature block')
fig.colorbar(im, ax=ax, label='Mean |d|')
plt.tight_layout()
fig.savefig(config.FIGURE_DIR / '03_poder_discriminante.png', dpi=150, bbox_inches='tight')
plt.show()
